# Embedding Generation

This notebook generates dense vector representations for every CRMP chunk using the local `bge-m3` model served by Ollama. It validates the service, measures performance, and performs basic vector-quality checks.

The output is `data/embeddings/crmp_bge_m3_embeddings.json`.


## 1. Import the required libraries

Load file, timing, numerical, and HTTP utilities for local embedding generation.


In [1]:
from pathlib import Path
import json
import time
import numpy as np
import requests

## 2. Define paths and model configuration

Set the chunk input, embedding output, Ollama endpoint, and embedding model.


In [2]:
# Run this notebook from notebooks/ so the project root is its parent.
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_chunks.json"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "embeddings"
    / "crmp_bge_m3_embeddings.json"
)

OLLAMA_URL = "http://localhost:11434/api/embed"

EMBEDDING_MODEL = "bge-m3"

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Model:  {EMBEDDING_MODEL}")


Input:  c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_chunks.json
Output: c:\Users\user\Documents\GitHub\legal-rag-pt\data\embeddings\crmp_bge_m3_embeddings.json
Model:  bge-m3


## 3. Load the chunk dataset

Read all retrieval chunks created by the previous stage.


In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]

print(f"Chunks carregados: {len(chunks)}")

Chunks carregados: 1617


## 4. Inspect a source chunk

Review the identifier and text that will be sent to the embedding model.


In [5]:
print(chunks[0]["id"])
print()
print(chunks[0]["text"])

crmp_a_1_chunk_001

Artigo A/1.º
Objeto do código
1 – O presente código consagra as disposições regulamentares com eficácia externa em
vigor na área do Município do Porto nos seguintes domínios:
a) Urbanismo;
b) Ambiente;
c) Gestão do espaço público;
d) Intervenção municipal sobre o exercício de atividades privadas;
e) Gestão de recursos;
f) Taxas e outras receitas municipais;
g) Fiscalização e sancionamento de infrações.
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições
regulamentares complementares ao presente código, nele devidamente referenciadas.


## 5. Define the Ollama embedding client

Create a helper that requests an embedding and validates the HTTP response.


In [6]:
def get_embedding(
    text,
    model=EMBEDDING_MODEL,
    url=OLLAMA_URL
):
    response = requests.post(
        url,
        json={
            "model": model,
            "input": text
        },
        timeout=120
    )

    # Surface Ollama errors immediately instead of accepting an invalid payload.
    response.raise_for_status()

    result = response.json()

    return result["embeddings"][0]


## 6. Test the embedding service

Send a short legal sentence to confirm model availability and vector dimensionality.


In [7]:
test_text = (
    "O Município do Porto prossegue "
    "o interesse público."
)

embedding = get_embedding(test_text)

print(f"Dimensão: {len(embedding)}")
print(embedding[:10])

Dimensão: 1024
[-0.0019888966, 0.024731938, -0.009913574, -0.0077872616, 0.002033821, 0.008367505, 0.036170207, -0.029158428, -0.009886227, -0.02159177]


## 7. Benchmark one real chunk

Measure embedding latency and dimensions using a representative corpus item.


In [8]:
test_chunk = chunks[0]

# perf_counter provides a monotonic high-resolution duration measurement.
start = time.perf_counter()

embedding = get_embedding(
    test_chunk["text"]
)

elapsed = time.perf_counter() - start

print(f"Chunk: {test_chunk['id']}")
print(f"Dimensão: {len(embedding)}")
print(f"Tempo: {elapsed:.3f} s")


Chunk: crmp_a_1_chunk_001
Dimensão: 1024
Tempo: 14.162 s


## 8. Define cosine similarity

Implement the similarity measure used for a small semantic sanity check.


In [9]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    # Normalize the dot product so similarity is independent of vector magnitude.
    return np.dot(a, b) / (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )


## 9. Compare related and unrelated texts

Verify that semantically related legal queries receive more similar vectors.


In [10]:
text_1 = (
    "Quais são as regras relativas "
    "ao estacionamento?"
)

text_2 = (
    "Regulamentação do trânsito "
    "e estacionamento."
)

text_3 = (
    "Proteção e tratamento de dados pessoais."
)

emb_1 = get_embedding(text_1)
emb_2 = get_embedding(text_2)
emb_3 = get_embedding(text_3)

print(
    "Relacionados:",
    cosine_similarity(emb_1, emb_2)
)

print(
    "Não relacionados:",
    cosine_similarity(emb_1, emb_3)
)

Relacionados: 0.7160155093349745
Não relacionados: 0.35828999187226


## 10. Generate all embeddings

Embed every chunk while recording per-item processing time and progress.


In [11]:
embeddings = []

start_total = time.perf_counter()

# Process sequentially to keep local resource usage predictable.
for index, chunk in enumerate(
    chunks,
    start=1
):

    start = time.perf_counter()

    vector = get_embedding(
        chunk["text"]
    )

    elapsed = time.perf_counter() - start

    embeddings.append({
        "id": chunk["id"],
        "embedding": vector,
        "embedding_model": EMBEDDING_MODEL,
        "embedding_dimension": len(vector),
        "embedding_time_seconds": elapsed
    })

    print(
        f"[{index}/{len(chunks)}] "
        f"{chunk['id']} "
        f"- {elapsed:.3f}s"
    )

total_elapsed = (
    time.perf_counter()
    - start_total
)

print()
print(
    f"Tempo total: "
    f"{total_elapsed:.2f}s"
)


[1/1617] crmp_a_1_chunk_001 - 1.034s
[2/1617] crmp_a_2_chunk_001 - 0.807s
[3/1617] crmp_a_1_1_chunk_001 - 0.818s
[4/1617] crmp_a_1_2_chunk_001 - 0.874s
[5/1617] crmp_a_1_3_chunk_001 - 0.970s
[6/1617] crmp_a_1_4_chunk_001 - 0.854s
[7/1617] crmp_a_1_5_chunk_001 - 0.854s
[8/1617] crmp_a_1_6_chunk_001 - 1.017s
[9/1617] crmp_a_1_7_chunk_001 - 0.931s
[10/1617] crmp_a_2_1_chunk_001 - 1.221s
[11/1617] crmp_a_2_2_chunk_001 - 0.819s
[12/1617] crmp_a_2_3_chunk_001 - 0.840s
[13/1617] crmp_a_2_4_chunk_001 - 0.957s
[14/1617] crmp_a_2_5_chunk_001 - 0.841s
[15/1617] crmp_a_2_6_chunk_001 - 0.836s
[16/1617] crmp_a_2_7_chunk_001 - 0.842s
[17/1617] crmp_a_2_8_chunk_001 - 0.843s
[18/1617] crmp_a_2_9_chunk_001 - 0.903s
[19/1617] crmp_a_2_10_chunk_001 - 0.941s
[20/1617] crmp_a_2_11_chunk_001 - 0.937s
[21/1617] crmp_a_2_12_chunk_001 - 0.875s
[22/1617] crmp_a_2_13_chunk_001 - 0.852s
[23/1617] crmp_a_2_13_a_chunk_001 - 0.930s
[24/1617] crmp_a_2_14_chunk_001 - 0.844s
[25/1617] crmp_a_2_15_chunk_001 - 0.825s
[26/

## 11. Summarize embedding latency

Calculate total, mean, minimum, and maximum generation times.


In [12]:
times = [
    item["embedding_time_seconds"]
    for item in embeddings
]

print(
    f"Chunks processados: "
    f"{len(embeddings)}"
)

print(
    f"Tempo total: "
    f"{sum(times):.2f}s"
)

print(
    f"Tempo médio/chunk: "
    f"{np.mean(times):.4f}s"
)

print(
    f"Mediana: "
    f"{np.median(times):.4f}s"
)

print(
    f"Min: "
    f"{np.min(times):.4f}s"
)

print(
    f"Max: "
    f"{np.max(times):.4f}s"
)

Chunks processados: 1617
Tempo total: 1353.86s
Tempo médio/chunk: 0.8373s
Mediana: 0.8266s
Min: 0.5641s
Max: 8.2662s


## 12. Calculate throughput

Estimate how many chunks are embedded per second.


In [13]:
chunks_per_second = (
    len(embeddings)
    / sum(times)
)

print(
    f"Throughput: "
    f"{chunks_per_second:.2f} chunks/s"
)

Throughput: 1.19 chunks/s


## 13. Save embeddings and metadata

Persist vectors, model details, dimensions, and performance statistics.


In [14]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

output = {
    "document": data["document"],
    "source_file": data["source_file"],

    "embedding_model": EMBEDDING_MODEL,

    "embedding_dimension": (
        len(embeddings[0]["embedding"])
        if embeddings
        else None
    ),

    "num_embeddings": len(embeddings),

    "performance": {
        "total_seconds": sum(times),
        "mean_seconds_per_chunk": float(
            np.mean(times)
        ),
        "median_seconds_per_chunk": float(
            np.median(times)
        ),
        "chunks_per_second": float(
            len(embeddings) / sum(times)
        )
    },

    "embeddings": embeddings
}

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False  # Preserve Portuguese metadata and chunk text.
    )

print(
    f"Ficheiro criado: "
    f"{OUTPUT_FILE}"
)


Ficheiro criado: c:\Users\user\Documents\GitHub\legal-rag-pt\data\embeddings\crmp_bge_m3_embeddings.json


## 14. Reload the embedding file

Verify that the generated JSON can be read successfully.


In [15]:
with open(
    OUTPUT_FILE,
    "r",
    encoding="utf-8"
) as f:

    saved = json.load(f)

print(saved["embedding_model"])
print(saved["embedding_dimension"])
print(saved["num_embeddings"])

bge-m3
1024
1617


## 15. Inspect a saved vector

Check the first stored identifier, vector length, and sample values.


In [16]:
first = saved["embeddings"][0]

print(first["id"])
print(len(first["embedding"]))
print(first["embedding"][:5])

crmp_a_1_chunk_001
1024
[-0.015608626, 0.037910216, -0.0060649877, -0.02934645, -0.009803567]


## 16. Analyze vector norms

Measure vector magnitudes as a basic consistency check across the dataset.


In [17]:
norms = [
    np.linalg.norm(
        item["embedding"]
    )
    for item in embeddings
]

print(
    f"Norma média: "
    f"{np.mean(norms):.6f}"
)

print(
    f"Min: "
    f"{np.min(norms):.6f}"
)

print(
    f"Max: "
    f"{np.max(norms):.6f}"
)

Norma média: 1.000000
Min: 0.999999
Max: 1.000001
